# 02 · Train and evaluate EyeGazeNet v2 (M3)

Prerequisite: notebook 01 has produced `MyDrive/eyegaze/manifest.parquet`.

* Runtime: **GPU** (Runtime → Change runtime type). Uses Colab's preinstalled CUDA PyTorch.
* Only the frames listed in the manifest are extracted to the local disk (resumable).
* Runs are written to Drive; if the session dies, re-run all cells and training resumes from `last.pt`.

In [ ]:
# --- Settings -------------------------------------------------------------
REPO_URL = "https://github.com/viettt0119/core_nckh.git"   # private repo: https://<TOKEN>@github.com/...
BRANCH = "claude/exciting-johnson-bfrv9s"
ARCHIVE = "/content/drive/MyDrive/archive.zip"
DRIVE_OUT = "/content/drive/MyDrive/eyegaze"
LOCAL_DATA = "/content/gazecapture"
CONFIG = "configs/base.yaml"          # E1; see research/configs/ for E0/E3 variants
EXTRA_OVERRIDES = []                  # e.g. ["optim.epochs=10", "data.limit_train=200000"]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash -s "$REPO_URL" "$BRANCH"
set -euo pipefail
if [ ! -d /content/core_nckh ]; then git clone -q --branch "$2" "$1" /content/core_nckh; else git -C /content/core_nckh pull -q; fi
pip -q install -e "/content/core_nckh/research[data]"
python -c "import torch, eyegaze; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
import os, subprocess, json, yaml
MANIFEST = f"{DRIVE_OUT}/manifest.parquet"
assert os.path.isfile(MANIFEST), "Run notebook 01 first"
subprocess.run(["eyegaze", "data", "extract", "--zip", ARCHIVE, "--out", LOCAL_DATA,
                "--only-manifest", MANIFEST], check=True)

In [ ]:
os.chdir("/content/core_nckh/research")
run_name = yaml.safe_load(open(CONFIG))["run_name"]
run_dir = f"{DRIVE_OUT}/runs/{run_name}"
overrides = [
    f"data.manifest={MANIFEST}",
    f"data.frames_root={LOCAL_DATA}",
    f"output_dir={DRIVE_OUT}/runs",
    f"data.num_workers={os.cpu_count()}",
    *EXTRA_OVERRIDES,
]
cmd = ["eyegaze", "train", "--config", CONFIG]
for o in overrides:
    cmd += ["--set", o]
if os.path.isfile(f"{run_dir}/last.pt"):
    cmd.append("--resume")
subprocess.run(cmd, check=True)

In [ ]:
subprocess.run(["eyegaze", "evaluate", "--run", run_dir, "--manifest", MANIFEST,
                "--frames-root", LOCAL_DATA, "--split", "test", "--svr",
                "--workers", str(os.cpu_count())], check=True)
print(json.dumps(json.load(open(f"{run_dir}/eval_test.json"))["accuracy"]["frame"], indent=2))

## Ablation table (E0/E3)

After training and evaluating several configs (change `CONFIG` above and re-run), print the comparison:

In [ ]:
import glob
runs = sorted(glob.glob(f"{DRIVE_OUT}/runs/*"))
print(subprocess.run(["eyegaze", "ablation-table", *runs, "--baseline", "base"],
                     capture_output=True, text=True, check=True).stdout)